In [ ]:
import xarray as xr
import os
import geopandas as gpd
import numpy as np
import xarray as xr
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from multiprocessing import Pool

In [4]:
def clean_nc_file_dim_time(file, districts, vars):
    with xr.open_dataset(file) as ds:
        lons = ds["lon"].values
        lats = ds["lat"].values

        # Create GeoDataFrame of points
        point_gdf = gpd.GeoDataFrame(
            geometry=gpd.points_from_xy(lons, lats),
            crs="EPSG:4326"
        )

        # Spatial join - extract county code
        joined = gpd.sjoin(point_gdf, districts, how="left", predicate="within")

        # Extract date
        dates = ds["time"].values.astype("datetime64[D]").astype(str)

        # Add back to the netcdf
        ds["region"] = xr.DataArray(joined["dt_adcode"].to_numpy(), dims="time")
        ds["date"] = xr.DataArray(dates, dims="time")
        print("added region and time variable")

        # Filter valid rows
        cat1 = pd.Categorical(ds["region"].values)
        cat2 = pd.Categorical(ds["date"].values)
        mask = (cat1.codes >= 0) & (cat2.codes >= 0)
        ds_masked = ds.isel(time=mask)  
        print("masked")

        # Created combined group of region and time
        region_vals = ds_masked["region"].values
        date_vals = ds_masked["date"].values
        combined_labels = [f"{r}_{d}" for r, d in zip(region_vals, date_vals)]
        ds_masked["combined_group"] = xr.DataArray(combined_labels, dims=["time"])
        print("assigned combined group")

        # Select variables to keep
        ds_selected = ds_masked[vars]
        print("selected target variables")

        # Convert to dataframe
        df_temp = ds_selected.to_dataframe()
        print("converted to dataframe")

        return df_temp
    
# load all files
os.chdir("/Users/anorawu/Team MG Dropbox/Wanru Wu/Cloudseeding/data/SSF")

path = os.getcwd() 

# Get the list of all files and directories 
all_files = os.listdir(path) 
data_files_terra = [file for file in all_files if file.find("CERES_SSF_Terra-XTRK_Edition4A")!=-1]

# Load the county-level district file
county_gdf = gpd.read_file(os.path.dirname(os.getcwd()) + "/district/district.shp")
county_gdf = county_gdf.to_crs("EPSG:4326")

variables_to_keep = ['CERES_SW_TOA_flux___upwards', 'combined_group']
df = clean_nc_file_dim_time(data_files_terra[0],county_gdf,variables_to_keep)
df


added region and time variable
masked
assigned combined group
selected target variables
converted to dataframe


,CERES_SW_TOA_flux___upwards,combined_group
time,,
2012-01-01 01:35:18.644781312,87.685104,232721_2012-01-01
2012-01-01 01:35:25.244785920,93.415451,232721_2012-01-01
2012-01-01 01:35:30.294808832,104.888229,232721_2012-01-01
2012-01-01 01:35:31.844790272,104.133980,232721_2012-01-01
2012-01-01 01:35:36.894772992,109.841537,231102_2012-01-01
...,...,...
2012-02-15 12:16:57.488707584,0.000000,232722_2012-02-15
2012-02-15 12:17:02.508676352,0.000000,232701_2012-02-15
2012-02-15 12:17:02.518694400,0.000000,232701_2012-02-15


In [8]:
df

,CERES_SW_TOA_flux___upwards,combined_group
time,,
2012-01-01 01:35:18.644781312,87.685104,232721_2012-01-01T00:00:00.000000000
2012-01-01 01:35:25.244785920,93.415451,232721_2012-01-01T00:00:00.000000000
2012-01-01 01:35:30.294808832,104.888229,232721_2012-01-01T00:00:00.000000000
2012-01-01 01:35:31.844790272,104.133980,232721_2012-01-01T00:00:00.000000000
2012-01-01 01:35:36.894772992,109.841537,231102_2012-01-01T00:00:00.000000000
...,...,...
2012-02-15 12:16:57.488707584,0.000000,232722_2012-02-15T00:00:00.000000000
2012-02-15 12:17:02.508676352,0.000000,232701_2012-02-15T00:00:00.000000000
2012-02-15 12:17:02.518694400,0.000000,232701_2012-02-15T00:00:00.000000000


In [14]:
df2 = df.iloc[1:3]
df2
df2.groupby("combined_group").mean()

,CERES_SW_TOA_flux___upwards
combined_group,
232721_2012-01-01T00:00:00.000000000,99.15184


In [206]:
import numpy as np
import xarray as xr

ds2 = xr.Dataset(
    {
        "temperature_1": ("time", 20 * np.random.rand(4)),  
        "temperature_2": ("time", 20 * np.random.rand(4)),
        "pressure": (("time", "other_dim"), np.array([1, 2, 3, 4 ,5 ,6,7,8,9,10,11,None]).reshape(4, 3)),
    },
    coords={"time": [10, 20, 30, 40], "other_dim": [1, 2, 3]},
)

# Grouping variable (should also be shape (2,))
l = xr.DataArray(np.array([None,0,0,None]), dims=["time"])
ds2["temperature_3"] = l
l2 = xr.DataArray(np.array([1,3,3,None]), dims=["time"])
ds2 = ds2.assign(temperature_4=l2)

# Encode two variables into categories (integers)
cat1 = pd.Categorical(ds2["temperature_3"].values)
cat2 = pd.Categorical(ds2["temperature_4"].values)
mask = (cat1.codes >= 0) & (cat2.codes >= 0)
ds3 = ds2.isel(time=mask) 

label_group = [f"{a}_{b}" for (a,b) in zip(ds3["temperature_3"].values,ds3["temperature_4"].values)]
ds3['combined_group'] = xr.DataArray(label_group, dims=["time"])

ds3 = ds3.to_dataframe()
ds3.groupby(['combined_group']).mean().reset_index()

,combined_group,temperature_1,temperature_2,pressure,temperature_3,temperature_4
0,0_3,4.317254,12.5413,6.5,0.0,3.0


In [22]:
c = pd.DataFrame({'a':[1,None,2,None],'b':[1,2,1,2]})
c

,a,b
0,1.0,1
1,NaN,2
2,2.0,1
3,NaN,2


In [23]:
c.groupby('b').mean()

,a
b,
1,1.5
2,NaN


In [37]:
with xr.open_dataset("/Users/anorawu/Library/CloudStorage/Box-Box/ClimateCompensationData/input/CIL/temps/adding_up_euler_ramsey_eta1.34_rho0.0001_scc.nc4") as ds:
    ds
ds_subset = ds.sel(ssp="SSP2",model="IIASA GDP",rcp="ssp245",weitzman_parameter="0.1", fair_aggregation = "ce")
ds_subset.scc.values

array([[63.74587738]])

In [17]:
with xr.open_dataset("/Users/anorawu/Library/CloudStorage/Box-Box/ClimateCompensationData/input/CIL/ir_damages/region_pc_energy_damages_more_years.nc", engine="netcdf4") as ds:
    ds
ds


<xarray.Dataset>
Dimensions:  (rcp: 2, region: 24378, ssp: 2, year: 13)
Coordinates:
    model    object ...
  * rcp      (rcp) object 'rcp45' 'rcp85'
  * region   (region) object 'ABW' 'AFG.1.12' ... 'ZWE.9.R19a261b63a503c9c'
  * ssp      (ssp) object 'SSP2' 'SSP3'
  * year     (year) int64 2030 2050 2089 2090 2091 ... 2095 2096 2097 2098 2099
Data variables:
    damages  (rcp, ssp, region, year) float32 ...

In [ ]:
import xarray as xr

# Paths ----
dampath="" + "AMEL_m0.zarr"
socpath= "/project/cil/gcp/integration/float32/dscim_input_data/econvars/zarrs/integration-econ-bc39.zarr"

# Combined damages time series ----
# Per capita damages
ds = xr.open_zarr(dampath)
dssubset=ds["summed_delta"].sel(eta=1.0).rename("damages")

# Population
soc = xr.open_zarr(socpath)
socsubset=soc["pop"].rename("damages")
socgdpsubset=soc["gdp"].rename("damages")

# Total IR damages
ir_total_damages = socsubset * dssubset